# Dedicated Per-Drug RF (Fast — RandomizedSearchCV + Checkpoints)

**Loads preprocessed data from 07-01.** Runs RandomizedSearchCV with subsampling for speed, saves checkpoints after each drug.

| Feature | Detail |
|---|---|
| **Grid search** | RandomizedSearchCV, n_iter=20, cv=2, subsampled to max 5000 samples |
| **Retrain** | Best params retrained on FULL training data |
| **Checkpoints** | `rf_checkpoints/<drug>.joblib` — skip already-completed drugs on rerun |
| **Comparison** | LR (07-01) vs MLP (07-01) vs RF (07-03) |

Compatible with Google Colab.

In [ ]:
!pip install maldideepkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, pickle, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
SEED = 42
np.random.seed(SEED)

RF_PARAM_GRID = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [2, 5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}

THRESHOLDS = np.linspace(0.05, 0.95, 91)
MAX_GS_SAMPLES = 5000
N_ITER = 20

print(f"Params: n_iter={N_ITER}, max_grid_samples={MAX_GS_SAMPLES}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_dedicated_lr_mlp_rf")
OUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUT_DIR / "rf_checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Load saved data from 07-01
SAVE_DIR = Path("./results_dedicated_lr_mlp")
assert SAVE_DIR.exists(), "Run 07-01 save cell first!"

DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]

In [ ]:
# ── 1. LOAD PREPROCESSED DATA FROM 07-01 ──

loaded = {}
for drug in DRUGS_10:
    safe_name = drug.replace(" ", "_").replace("-", "_")
    p = SAVE_DIR / f"data_{safe_name}.pkl"
    if not p.exists():
        print(f"  SKIP {drug}: no saved data")
        continue
    with open(p, "rb") as f:
        loaded[drug] = pickle.load(f)
    n_tr = len(loaded[drug]["y_train"])
    n_te = len(loaded[drug]["y_test"])
    n_tot = n_tr + n_te
    print(f"  {drug:35s}  {n_tot:>6d} total  (train={n_tr}, test={n_te})")

# Load LR + MLP results from 07-01 for final comparison
try:
    with open(SAVE_DIR / "lr_results.pkl", "rb") as f:
        lr_07_01 = pickle.load(f)
    with open(SAVE_DIR / "mlp_results.pkl", "rb") as f:
        mlp_07_01 = pickle.load(f)
    print(f"\nLoaded LR ({len(lr_07_01)}) + MLP ({len(mlp_07_01)}) results from 07-01")
except FileNotFoundError:
    lr_07_01, mlp_07_01 = {}, {}
    print("\nWARNING: LR/MLP results not found -- comparison will be incomplete")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2. TRAIN RF PER DRUG  (RandomizedSearchCV + subsampling + checkpoints)
# ═══════════════════════════════════════════════════════════════════════════

rf_results = {}

for drug in DRUGS_10:
    if drug not in loaded:
        print(f"  SKIP {drug}: not loaded")
        continue

    safe_name = drug.replace(" ", "_").replace("-", "_")
    ckpt_path = CHECKPOINT_DIR / f"{safe_name}.joblib"

    # Checkpoint: skip if already done
    if ckpt_path.exists():
        print(f"  {drug:35s}  CHECKPOINT exists, skipping")
        continue

    print(f"\n{'='*60}")
    print(f"  {drug}")
    print(f"{'='*60}")

    d = loaded[drug]
    X_train_pp, y_train = d["X_train_pp"], d["y_train"]
    X_val_pp, y_val = d["X_val_pp"], d["y_val"]
    X_test_pp, y_test = d["X_test_pp"], d["y_test"]

    # Subsampled RandomizedSearchCV
    n_gs = min(len(X_train_pp), MAX_GS_SAMPLES)
    idx_gs = np.random.choice(len(X_train_pp), n_gs, replace=False)
    X_gs, y_gs = X_train_pp[idx_gs], y_train[idx_gs]
    print(f"  Grid search on {n_gs} samples (subset of {len(X_train_pp)})")

    grid = RandomizedSearchCV(
        RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
        param_grid=RF_PARAM_GRID, n_iter=N_ITER, cv=2,
        scoring="balanced_accuracy", random_state=SEED, n_jobs=-1, verbose=0)
    grid.fit(X_gs, y_gs)
    print(f"  Best params: {grid.best_params_}")

    # Retrain on FULL training data
    rf = RandomForestClassifier(**grid.best_params_, oob_score=True,
                                 random_state=SEED, n_jobs=-1)
    rf.fit(X_train_pp, y_train)

    # CV threshold tuning (subsampled to 5000 for speed)
    n_cv = min(len(X_train_pp), MAX_GS_SAMPLES)
    idx_cv = np.random.choice(len(X_train_pp), n_cv, replace=False)
    X_cv, y_cv = X_train_pp[idx_cv], y_train[idx_cv]
    cv_proba = cross_val_predict(
        RandomForestClassifier(**grid.best_params_, random_state=SEED, n_jobs=-1),
        X_cv, y_cv, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_cv, cv_proba >= t) for t in THRESHOLDS])]

    # Evaluate on val and test
    def eval_one(X, y):
        proba = rf.predict_proba(X)[:, 1]
        preds = proba >= best_t
        return {
            "BalAcc": balanced_accuracy_score(y, preds),
            "AUC": roc_auc_score(y, proba),
            "Threshold": best_t,
            "Best_Param": str(grid.best_params_),
        }

    rf_results[drug] = {
        "Val": eval_one(X_val_pp, y_val),
        "Test": eval_one(X_test_pp, y_test),
    }
    print(f"    Val  BalAcc={rf_results[drug]['Val']['BalAcc']:.4f}  AUC={rf_results[drug]['Val']['AUC']:.4f}")
    print(f"    Test BalAcc={rf_results[drug]['Test']['BalAcc']:.4f}  AUC={rf_results[drug]['Test']['AUC']:.4f}")

    # Save checkpoint
    joblib.dump({"model": rf, "results": rf_results[drug]}, ckpt_path)
    print(f"    Checkpoint saved: {ckpt_path.name}")

print("\nRF training complete.")

In [ ]:
# ── Reload checkpointed results for already-completed drugs ──
for drug in DRUGS_10:
    if drug in rf_results: continue
    safe_name = drug.replace(" ", "_").replace("-", "_")
    ckpt_path = CHECKPOINT_DIR / f"{safe_name}.joblib"
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            ckpt = joblib.load(f)
        rf_results[drug] = ckpt["results"]
        print(f"  Loaded checkpoint: {drug}")

print(f"\nRF results for {len(rf_results)} drugs")

---
## Results: LR + MLP + RF

In [ ]:
# ── 3. BUILD COMPARISON TABLE ──

rows = []
for drug in DRUGS_10:
    if drug not in rf_results: continue
    row = {"Drug": drug}

    if drug in lr_07_01:
        row["LR"] = lr_07_01[drug]["Test"]["BalAcc"]
    else:
        row["LR"] = np.nan

    if drug in mlp_07_01:
        row["MLP"] = mlp_07_01[drug]["Test"]["BalAcc"]
    else:
        row["MLP"] = np.nan

    row["RF"] = rf_results[drug]["Test"]["BalAcc"]
    rows.append(row)

df_comp = pd.DataFrame(rows).set_index("Drug")
short_names = {d: d[:15] for d in df_comp.index}

print("\nTest Balanced Accuracy:")
print(df_comp.round(4).to_string())

# Gaps
for model in ["MLP", "RF"]:
    if model in df_comp.columns and "LR" in df_comp.columns:
        print(f"\nDelta ({model} - LR):")
        for drug in df_comp.index:
            delta = df_comp.loc[drug, model] - df_comp.loc[drug, "LR"]
            print(f"  {drug:35s}  {delta:+.4f}")

In [ ]:
# ── Heatmap: LR vs MLP vs RF ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3.5))

df_disp_ba = df_comp.copy(); df_disp_ba.index = [short_names[d] for d in df_disp_ba.index]
sns.heatmap(df_disp_ba.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy", "shrink": 0.8}, ax=ax1)
ax1.set_title("Balanced Accuracy — LR / MLP / RF", fontsize=12, fontweight="bold")

# AUC
auc_rows = []
for drug in DRUGS_10:
    if drug not in rf_results: continue
    auc_row = {"Drug": drug}
    if drug in lr_07_01: auc_row["LR"] = lr_07_01[drug]["Test"]["AUC"]
    else: auc_row["LR"] = np.nan
    if drug in mlp_07_01: auc_row["MLP"] = mlp_07_01[drug]["Test"]["AUC"]
    else: auc_row["MLP"] = np.nan
    auc_row["RF"] = rf_results[drug]["Test"]["AUC"]
    auc_rows.append(auc_row)
df_auc = pd.DataFrame(auc_rows).set_index("Drug")
df_auc.index = [short_names[d] for d in df_auc.index]

sns.heatmap(df_auc.T, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC", "shrink": 0.8}, ax=ax2)
ax2.set_title("AUC-ROC — LR / MLP / RF", fontsize=12, fontweight="bold")

fig.suptitle("Dedicated Per-Drug Models -- Aggregated 70/15/15 Test", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_all.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Bar chart: LR vs MLP vs RF ──
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(df_comp)); w = 0.25
colors = {"LR": "#1f77b4", "MLP": "#ff7f0e", "RF": "#2ca02c"}
for i, model in enumerate(["LR", "MLP", "RF"]):
    vals = [df_comp.loc[d, model] for d in df_comp.index]
    ax.bar(x + (i - 1) * w, vals, w, label=model, color=colors[model], edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([short_names[d] for d in df_comp.index], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy"); ax.set_title("LR vs MLP vs RF (Aggregated Test)")
ax.legend(fontsize=10); ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.grid(True, ls='--', lw=0.5, color='gray', alpha=0.5, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / "barchart_all.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Best RF hyperparameters ──
param_rows = []
for drug in DRUGS_10:
    if drug not in rf_results: continue
    row = {"Drug": drug[:20], "RF": rf_results[drug]["Test"]["Best_Param"]}
    param_rows.append(row)
df_params = pd.DataFrame(param_rows).set_index("Drug")
print("\nBest RF Hyperparameters:")
print(df_params.to_string())

# ── Save results ──
df_comp.to_csv(OUT_DIR / "comparison_lr_mlp_rf.csv")
df_params.to_csv(OUT_DIR / "rf_best_params.csv")

# Save RF results for federation
with open(OUT_DIR / "rf_results.pkl", "wb") as f:
    pickle.dump(rf_results, f)

print("\nSaved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")

---
**Done.** RF analysis complete. Results saved for federation.